# Domain 5: Context Management & Reliability
## TS 5.5: Design Human Review Workflows and Confidence Calibration
**Certification Focus**: CCAR-F Domain 5 (15% Exam Weight) | **Scenario Alignment**: Scenario 6 (Structured Data Extraction & Auditing)

---

### 🎯 Key Architectural Principles
1. **The Three Triage Tiers**:
   - **Tier 1: `AUTO_APPROVE`** (Confidence >= 0.90) -> processed automatically without human involvement.
   - **Tier 2: `STRATIFIED_AUDIT`** (Confidence >= 0.90 but selected in a 5–10% stratified random audit sample to check for silent drift).
   - **Tier 3: `MANDATORY_ESCALATION`** (Confidence < 0.80) -> routed directly to human specialists.
2. **Confidence Calibration**: Raw probabilities must be calibrated against historical ground truth to avoid overconfidence.
3. **Audit Trails**: Every routing decision must record its triage tier, confidence score, and audit flags.

In [ ]:
# Step 1: Environment & Setup
import random
import sys
from dataclasses import dataclass
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 5.5 Human Review Calibration.")

### ⚖️ Step 2: Calibrated Router Implementation
We implement the three-tier routing engine with stratified auditing.

In [ ]:
# Step 2: Calibrated Router Logic
class RoutingTier(Enum):
    AUTO_APPROVE = "AUTO_APPROVE"
    STRATIFIED_AUDIT = "STRATIFIED_AUDIT"
    MANDATORY_ESCALATION = "MANDATORY_ESCALATION"


@dataclass
class TriageDecision:
    doc_id: str
    confidence: float
    doc_type: str
    routing_tier: RoutingTier
    rationale: str


class ConfidenceRouter:
    def __init__(self, high_confidence_threshold: float = 0.90, audit_rate: float = 0.10, random_seed: int = 42) -> None:
        self.high_threshold = high_confidence_threshold
        self.audit_rate = audit_rate
        random.seed(random_seed)

    def route_item(self, doc_id: str, confidence: float, doc_type: str) -> TriageDecision:
        if confidence < self.high_threshold:
            return TriageDecision(
                doc_id=doc_id,
                confidence=confidence,
                doc_type=doc_type,
                routing_tier=RoutingTier.MANDATORY_ESCALATION,
                rationale=f"Confidence {confidence:.2f} is below high-confidence threshold ({self.high_threshold:.2f})."
            )

        # High confidence item: test for stratified audit sample
        is_audited = (random.random() < self.audit_rate)
        if is_audited:
            return TriageDecision(
                doc_id=doc_id,
                confidence=confidence,
                doc_type=doc_type,
                routing_tier=RoutingTier.STRATIFIED_AUDIT,
                rationale="Selected for stratified quality audit sample."
            )

        return TriageDecision(
            doc_id=doc_id,
            confidence=confidence,
            doc_type=doc_type,
            routing_tier=RoutingTier.AUTO_APPROVE,
            rationale="Confidence meets threshold; auto-approved."
        )

print(" Calibrated ConfidenceRouter initialized.")

### 🧪 Step 3: Interactive Simulation
1. Low confidence -> Mandatory Escalation.
2. High confidence -> Auto-approved or Stratified Audit.

In [ ]:
# Step 3: Simulation
router = ConfidenceRouter(high_confidence_threshold=0.90, audit_rate=0.10, random_seed=42)

# Case 1: Low confidence document
dec_low = router.route_item("doc_101", 0.65, "invoice")
print("Doc 101 (0.65) Tier:", dec_low.routing_tier.value)
print("Rationale:", dec_low.rationale)

# Case 2: High confidence document
dec_high = router.route_item("doc_102", 0.98, "invoice")
print("\nDoc 102 (0.98) Tier:", dec_high.routing_tier.value)
print("Rationale:", dec_high.rationale)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Low confidence document maps to `MANDATORY_ESCALATION`.
2. High confidence document maps to `AUTO_APPROVE` or `STRATIFIED_AUDIT`.
3. Routing tiers conform to the CCAR-F calibration taxonomy.

In [ ]:
# Step 4: Verification Checks
assert dec_low.routing_tier == RoutingTier.MANDATORY_ESCALATION
assert dec_high.routing_tier in (RoutingTier.AUTO_APPROVE, RoutingTier.STRATIFIED_AUDIT)

print(" All verification assertions passed! Confidence calibration verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Low confidence (<0.80) requires **mandatory escalation** to human specialists.
- **Rule 2**: High confidence (>=0.90) items must still be subjected to a **stratified random audit sample** (5–10%) to catch silent drift.
- **Rule 3**: Store confidence and rationale in the audit trail of every automated triage decision.